# 3-D batches and performance

Every notebook so far has transformed one `(n_radial, n_angular)` array at a
time. Real acquisitions rarely come one image at a time -- an MRI volume is a
stack of slices, a dynamic scan is a stack of time points -- so
`pypft.forward_pft`/`pypft.inverse_pft` (and `pypft.BaseSignal`'s step
methods) also accept a 3-D `(n_radial, n_angular, batch)` array: the same two
polar axes as before, plus one trailing batch axis (`pypft.Axis.BATCH`,
`pypft.DEFAULT_BATCH_AXIS`). No new numerical kernel is involved -- the
angular DFT/IDFT and the discrete Hankel transform were already able to
operate along one named axis of an otherwise arbitrary-rank array, so a
batch of signals transforms in one call instead of a Python loop over many
calls.

In [ ]:
import time

import numpy as np

import pypft

## The batch axis is always last

PyPFT's own axis convention (`pypft.Axis`) fixes the radial axis at position
0 and the angular axis at position 1; a batch, when present, is always the
*last* axis -- position 2 of a 3-D array, which is exactly what
`pypft.DEFAULT_BATCH_AXIS` (`-1`) already names. There is nothing to
configure: passing a 3-D array is enough.

In [ ]:
grid = pypft.PolarGrid(n_radial=96, n_angular=15, R=40.0)

# A small batch of Gaussians, each a different width -- f(r) = exp(-a * r^2).
widths = np.array([0.5, 1.0, 2.0, 4.0])
f_batch = np.exp(-widths * grid.r.T[..., np.newaxis] ** 2)
f_batch.shape

## Batching is exact, not approximate

Transforming the whole batch in one call gives *exactly* the same result as
transforming each slice on its own -- batching is purely a performance and
convenience feature, never a different computation:

In [ ]:
F_batch = pypft.forward_pft(f=f_batch, grid=grid)

looped = np.stack(
    [pypft.forward_pft(f=f_batch[..., b], grid=grid) for b in range(f_batch.shape[-1])],
    axis=-1,
)
float(np.abs(F_batch - looped).max())

## Domain objects batch too

`pypft.BaseSignal` and its subclasses carry the batch axis along for free --
every step method preserves it, so walking the chain on a batch is no
different from walking it on a single image:

In [ ]:
signal = pypft.SpacePolarSignal(values=f_batch, grid=grid)
walked = signal.to(pypft.Domain.FREQUENCY_POLAR)

walked.values.shape, float(np.abs(walked.values - F_batch).max())

## The batch axis really must be last

Since PyPFT's layout fixes the batch axis at the end, asking for it anywhere
else -- or asking for one on a plain 2-D array, which has no batch axis at
all -- is rejected immediately, the same way any other malformed argument
is:

In [ ]:
try:
    pypft.forward_pft(f=f_batch, grid=grid, batch_axis=0)
except ValueError as exc:
    print(exc)

## Why batching is worth it

A single batched call is also faster than a Python loop over the same
slices, since the per-harmonic step (`pypft.transform.scaled_hankel`) can
apply one harmonic's kernel to the *whole* batch at once instead of paying
Python-level call overhead once per slice. The effect grows with the batch
size -- here with a batch large enough to make the difference clear:

In [ ]:
rng = np.random.default_rng(0)
big_batch = rng.standard_normal((grid.n_radial, grid.n_angular, 64))

start = time.perf_counter()
pypft.forward_pft(f=big_batch, grid=grid)
batched_seconds = time.perf_counter() - start

start = time.perf_counter()
for b in range(big_batch.shape[-1]):
    pypft.forward_pft(big_batch[..., b], grid)
looped_seconds = time.perf_counter() - start

batched_seconds, looped_seconds

## Where to go next

Batching only changes *how many* signals a single call transforms, never
*what* it computes -- every earlier notebook's accuracy figures and
analytical properties apply unchanged to each slice of a batch. The next
notebook turns to visualization: plotting a signal in whichever domain it
currently occupies.